# NSCI 270: ABCD Longitudinal Analysis

This notebook examines whether baseline diet quality moderates the association between changes in sleep duration and changes in working-memory-related brain activation in adolescents.

Data from ABCD sessions 04A and 06A will be used.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 30)

## 1. Data Location

Locate the ABCD phenotype files and confirm that the required datasets are available.

In [ ]:
data_dir = Path(
    "/hickory/proj/dcmonroelab/nimhda/ABCD/abcd/rawdata/phenotype"
)

files = [
    "ph_y_mctq.tsv",
    "ph_y_bkfs.tsv",
    "mr_y_tfmri__nback__2bv0b__dsk.tsv"
]

print("Data directory exists:", data_dir.exists())

for filename in files:
    matches = list(data_dir.rglob(filename))
    print(f"{filename}: {len(matches)} match(es)")
    for path in matches[:3]:
        print(" ", path)

## 2. Load and Inspect Datasets

Load the sleep, diet, and N-back fMRI datasets. Check their dimensions, participant identifiers, sessions, and available variables before cleaning.

In [ ]:
sleep = pd.read_csv(data_dir / "ph_y_mctq.tsv", sep="\t", low_memory=False)
diet = pd.read_csv(data_dir / "ph_y_bkfs.tsv", sep="\t", low_memory=False)
fmri = pd.read_csv(
    data_dir / "mr_y_tfmri__nback__2bv0b__dsk.tsv",
    sep="\t",
    low_memory=False
)

for name, df in [("Sleep", sleep), ("Diet", diet), ("fMRI", fmri)]:
    print(f"\n{name}: {df.shape}")
    print("First columns:", df.columns[:10].tolist())

In [ ]:
for name, df in [("Sleep", sleep), ("Diet", diet), ("fMRI", fmri)]:
    print(f"\n{name}")
    print("ID columns:", [c for c in df.columns if "subject" in c.lower() or "session" in c.lower()])

    if "session_id" in df.columns:
        print("Sessions:")
        print(df["session_id"].value_counts().head(10))

## 3. Select Variables and Match Sessions

Select the variables needed for the study and merge sleep, diet, and fMRI data using participant and session IDs. Keep only participants with data at both sessions.

In [ ]:
sessions = ["ses-04A", "ses-06A"]
ids = ["participant_id", "session_id"]

sleep_cols = ids + ["ph_y_mctq__sleep_dur"]

diet_cols = ids + [
    "ph_y_bkfs__othfruit_cup",
    "ph_y_bkfs__vegnopota_cup",
    "ph_y_bkfs__whlgrain_oz",
    "ph_y_bkfs__addsugar_tsp"
]

roi_cols = [
    f"mr_y_tfmri__nback__2bv0b__dsk__{roi}__{hemi}_beta"
    for roi in ["cmfrt", "rmfrt", "sfrt"]
    for hemi in ["lh", "rh"]
]

print("Missing sleep columns:", set(sleep_cols) - set(sleep.columns))
print("Missing diet columns:", set(diet_cols) - set(diet.columns))
print("Missing fMRI columns:", set(ids + roi_cols) - set(fmri.columns))

In [ ]:
sleep_sub = sleep.loc[
    sleep["session_id"].isin(sessions), sleep_cols
].copy()

diet_sub = diet.loc[
    diet["session_id"].isin(sessions), diet_cols
].copy()

fmri_sub = fmri.loc[
    fmri["session_id"].isin(sessions), ids + roi_cols
].copy()

merged = sleep_sub.merge(
    diet_sub, on=ids, how="inner", validate="one_to_one"
).merge(
    fmri_sub, on=ids, how="inner", validate="one_to_one"
)

print("Matched observations:", len(merged))
print("Matched participants:", merged["participant_id"].nunique())

session_counts = merged.groupby("participant_id")["session_id"].nunique()

matched_ids = session_counts[session_counts == 2].index

longitudinal = merged[
    merged["participant_id"].isin(matched_ids)
].copy()

print("Participants with both sessions:", longitudinal["participant_id"].nunique())
print("Total longitudinal observations:", len(longitudinal))

## 4. Data Quality Assessment

Check missing values and non-numeric entries in the matched longitudinal dataset before applying exclusion criteria.

In [ ]:
analysis_cols = [
    "ph_y_mctq__sleep_dur"
] + diet_cols[2:] + roi_cols

# Convert analysis variables to numeric values
longitudinal[analysis_cols] = longitudinal[analysis_cols].apply(
    pd.to_numeric, errors="coerce"
)

# Count missing values for each variable
missing = longitudinal[analysis_cols].isna().sum()

print("Total observations:", len(longitudinal))
print("\nMissing values by variable:")
print(missing.to_string())

print("\nObservations with any missing values:")
print(longitudinal[analysis_cols].isna().any(axis=1).sum())

## 5. Imaging Quality Control

Identify the available N-back fMRI quality-control variables before applying exclusion criteria.

In [ ]:
qc_files = sorted(data_dir.glob("mr_y_qc*.tsv"))

print("Available QC files:")

for file in qc_files:
    print(file.name)

### 5.1 Identify Imaging QC Variables

Inspect the available imaging quality-control variables to determine which criteria should be used for N-back fMRI data.

In [ ]:
qc_names = [
    "mr_y_qc__incl.tsv",
    "mr_y_qc__raw__tfmri__nback.tsv",
    "mr_y_qc__post__man__fmri.tsv"
]

for filename in qc_names:
    qc = pd.read_csv(
        data_dir / filename,
        sep="\t",
        nrows=5
    )

    print(f"\n{filename}")
    print("Number of columns:", len(qc.columns))
    print("Columns:")
    print(qc.columns.tolist())

### 5.2 N-back Imaging Inclusion

Examine the N-back imaging inclusion indicator at both sessions before applying quality-control exclusions.

In [ ]:
qc_col = "mr_y_qc__incl__tfmri__nback_indicator"

qc = pd.read_csv(
    data_dir / "mr_y_qc__incl.tsv",
    sep="\t",
    usecols=ids + [qc_col]
)

qc_sub = qc[qc["session_id"].isin(sessions)].copy()

print("QC indicator values:")
print(qc_sub[qc_col].value_counts(dropna=False))

qc_merged = longitudinal.merge(
    qc_sub,
    on=ids,
    how="left",
    validate="one_to_one"
)

print("\nIndicator values in matched longitudinal data:")
print(qc_merged[qc_col].value_counts(dropna=False))

print("\nParticipants with indicator = 1 at both sessions:")

qc_merged[qc_col] = pd.to_numeric(
    qc_merged[qc_col], errors="coerce"
)

eligible = qc_merged[qc_merged[qc_col] == 1]

eligible_counts = eligible.groupby("participant_id")["session_id"].nunique()

print((eligible_counts == 2).sum())

## 6. Sleep Duration Screening

Retain participants with valid sleep duration (3–16 hours) at both sessions.

In [ ]:
# Keep participants passing N-back QC at both sessions
qc_ids = eligible_counts[eligible_counts == 2].index

qc_clean = qc_merged[
    qc_merged["participant_id"].isin(qc_ids)
].copy()

# Apply sleep duration screening
sleep_col = "ph_y_mctq__sleep_dur"

valid_sleep = qc_clean[sleep_col].between(3, 16)

sleep_ids = qc_clean.loc[valid_sleep].groupby(
    "participant_id"
)["session_id"].nunique()

sleep_ids = sleep_ids[sleep_ids == 2].index

sleep_clean = qc_clean[
    qc_clean["participant_id"].isin(sleep_ids)
].copy()

print("After imaging QC:", qc_clean["participant_id"].nunique())
print("After sleep screening:", sleep_clean["participant_id"].nunique())
print("Excluded by sleep screening:",
      qc_clean["participant_id"].nunique() -
      sleep_clean["participant_id"].nunique())

## 7. Dietary Data Screening

Exclude participants with missing dietary measurements at either session. All four dietary variables are required to calculate the Diet Quality Score.

In [ ]:
# Four dietary variables
diet_vars = diet_cols[2:]

# Check complete dietary data
valid_diet = sleep_clean[diet_vars].notna().all(axis=1)

# Keep participants with complete data at both sessions
diet_ids = sleep_clean.loc[valid_diet].groupby(
    "participant_id"
)["session_id"].nunique()

diet_ids = diet_ids[diet_ids == 2].index

cleaned = sleep_clean[
    sleep_clean["participant_id"].isin(diet_ids)
].copy()

print("After sleep screening:", sleep_clean["participant_id"].nunique())
print("After diet screening:", cleaned["participant_id"].nunique())
print("Total observations:", len(cleaned))
print("Excluded by diet screening:",
      sleep_clean["participant_id"].nunique() -
      cleaned["participant_id"].nunique())

## 8. Demographic and Motion Covariates

Locate the demographic and N-back head motion variables needed for the regression models.

In [ ]:
covariate_files = [
    "ab_g_dyn.tsv",
    "ab_g_stc.tsv",
    "mr_y_qc__mot.tsv"
]

for filename in covariate_files:
    path = data_dir / filename

    print(f"\n{filename}")
    print("File exists:", path.exists())

    if path.exists():
        columns = pd.read_csv(
            path, sep="\t", nrows=0
        ).columns

        matches = [
            c for c in columns
            if any(x in c.lower() for x in [
                "visit_age", "cohort_sex",
                "design_site", "nback__mot_mean"
            ])
        ]

        print("Relevant columns:", matches)

### 8.2 Merge Demographic and Motion Variables

Merge age, sex, study site, and N-back head motion into the cleaned longitudinal dataset. Check for missing values before statistical analysis.

In [ ]:
# Load demographic and motion variables
age_site = pd.read_csv(
    data_dir / "ab_g_dyn.tsv",
    sep="\t",
    usecols=ids + [
        "ab_g_dyn__visit_age",
        "ab_g_dyn__design_site"
    ]
)

sex = pd.read_csv(
    data_dir / "ab_g_stc.tsv",
    sep="\t",
    usecols=["participant_id", "ab_g_stc__cohort_sex"]
)

motion = pd.read_csv(
    data_dir / "mr_y_qc__mot.tsv",
    sep="\t",
    usecols=ids + [
        "mr_y_qc__mot__tfmri__nback__mot_mean"
    ]
)

# Merge covariates
cleaned = cleaned.merge(
    age_site, on=ids, how="left", validate="one_to_one"
)

cleaned = cleaned.merge(
    sex.drop_duplicates("participant_id"),
    on="participant_id", how="left", validate="many_to_one"
)

cleaned = cleaned.merge(
    motion, on=ids, how="left", validate="one_to_one"
)

# Check missing values
covariate_cols = [
    "ab_g_dyn__visit_age",
    "ab_g_dyn__design_site",
    "ab_g_stc__cohort_sex",
    "mr_y_qc__mot__tfmri__nback__mot_mean"
]

print("Participants:", cleaned["participant_id"].nunique())
print("Observations:", len(cleaned))

print("\nMissing covariate values:")
print(cleaned[covariate_cols].isna().sum())

## 9. Diet Quality Score (DQS)

Standardize four dietary variables using the baseline (ses-04A) mean and standard deviation. Calculate a composite Diet Quality Score for both sessions, with higher scores indicating better diet quality.

In [ ]:
diet_vars = diet_cols[2:]

# Baseline reference
baseline = cleaned[cleaned["session_id"] == "ses-04A"]

# Calculate z-scores using baseline mean and SD
for col in diet_vars:
    mean = baseline[col].mean()
    sd = baseline[col].std()

    cleaned[col + "_z"] = (cleaned[col] - mean) / sd

# Calculate Diet Quality Score
cleaned["DQS"] = (
    cleaned[diet_vars[0] + "_z"]
    + cleaned[diet_vars[1] + "_z"]
    + cleaned[diet_vars[2] + "_z"]
    - cleaned[diet_vars[3] + "_z"]
) / 4

# Summary by session
print(
    cleaned.groupby("session_id")["DQS"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(3)
)

## 10. Longitudinal Change Variables

Reshape the dataset so each participant has one row. Calculate changes in sleep duration, diet quality, and activation across the six frontal ROIs between ses-04A and ses-06A.

In [ ]:
# Variables measured at both sessions
change_vars = ["ph_y_mctq__sleep_dur", "DQS"] + roi_cols

# Convert longitudinal data to wide format
wide = cleaned.pivot(
    index="participant_id",
    columns="session_id",
    values=change_vars
)

# Flatten column names
wide.columns = [
    f"{variable}_{session}"
    for variable, session in wide.columns
]

# Calculate changes
wide["delta_sleep"] = (
    wide["ph_y_mctq__sleep_dur_ses-06A"]
    - wide["ph_y_mctq__sleep_dur_ses-04A"]
)

wide["delta_DQS"] = (
    wide["DQS_ses-06A"]
    - wide["DQS_ses-04A"]
)

for roi in roi_cols:
    wide["delta_" + roi] = (
        wide[roi + "_ses-06A"]
        - wide[roi + "_ses-04A"]
    )

# Check results
print("Participants:", len(wide))

delta_cols = (
    ["delta_sleep", "delta_DQS"]
    + ["delta_" + roi for roi in roi_cols]
)

print("\nMissing change values:")
print(wide[delta_cols].isna().sum())

print("\nSummary:")
print(wide[delta_cols].describe().loc[["mean", "std", "min", "max"]].round(3))

## 11. Prepare the Final Analysis Dataset

Add baseline demographic variables and N-back head motion measurements from both sessions. Verify that each participant has complete data for the planned regression models.

In [ ]:
# Baseline demographic variables
baseline_cov = cleaned[
    cleaned["session_id"] == "ses-04A"
][[
    "participant_id",
    "ab_g_dyn__visit_age",
    "ab_g_dyn__design_site",
    "ab_g_stc__cohort_sex"
]].copy()

baseline_cov = baseline_cov.set_index("participant_id")

# Add baseline covariates
wide = wide.join(baseline_cov)

# Add motion measurements from both sessions
motion_col = "mr_y_qc__mot__tfmri__nback__mot_mean"

motion_wide = cleaned.pivot(
    index="participant_id",
    columns="session_id",
    values=motion_col
)

motion_wide.columns = [
    "motion_04A", "motion_06A"
]

wide = wide.join(motion_wide)

# Calculate motion change
wide["delta_motion"] = (
    wide["motion_06A"] - wide["motion_04A"]
)

# Final checks
print("Participants:", len(wide))
print("Duplicate participants:", wide.index.duplicated().sum())
print("Total missing values:", wide.isna().sum().sum())

print("\nCovariate summary:")
print(
    wide[[
        "ab_g_dyn__visit_age",
        "motion_04A",
        "motion_06A",
        "delta_motion"
    ]].describe().round(3)
)

## 12. Final Verification and Data Export

Verify the session labels for head motion, confirm the final sample size, and save the analysis-ready dataset for subsequent statistical analyses.

In [ ]:
# Verify motion values using explicit session labels
motion_check = cleaned.pivot(
    index="participant_id",
    columns="session_id",
    values=motion_col
)

wide["motion_04A"] = motion_check["ses-04A"]
wide["motion_06A"] = motion_check["ses-06A"]

wide["delta_motion"] = wide["motion_06A"] - wide["motion_04A"]

# Final verification
assert len(wide) == 3672
assert wide.index.is_unique
assert wide.isna().sum().sum() == 0

print("Final participants:", len(wide))
print("Missing values:", wide.isna().sum().sum())
print("Motion variables verified.")

# Save only within the authorized Longleaf environment
output_dir = Path.home() / "NSCI270"
output_dir.mkdir(exist_ok=True)

output_file = output_dir / "ABCD_cleaned_longitudinal.csv"

wide.to_csv(output_file)

print("Saved to:", output_file)